# Integración de Bases de Graduados — Universidad Santo Tomás

**Objetivo.** Unificar en un único conjunto de datos las cuatro bases de graduados que
viven en `datos/`, normalizar sus campos y producir un dataset consolidado, limpio
y documentado, junto con un primer análisis descriptivo. **Actualización 2026:**
Bucaramanga se incorpora con sus hojas de **Pregrado y Posgrado** (con modalidad), y el
archivo **SPB-CM-CAU** aporta la Sede Principal de Bogotá, el Campus Medellín y los
centros VUAD, con una columna `sede` canónica.

## Fuentes de datos

| # | Archivo | Hoja(s) | Estructura |
|---|---------|---------|------------|
| 1 | `GRADUADOS UNIVERSIDAD SANTO TOMÁS SECCIONAL TUNJA.xls` | `Datos1` | 11 columnas (nombre desagregado) |
| 2 | `Base de Graduados Villavicencio ...Estadística.xls` | `Datos1` | 11 columnas (nombre desagregado) |
| 3 | `GRADUADOS SANTOTO BUCARAMANGA - POSGRADO.xlsx` | `PREGRADO`, `POSGRADO` | 5 columnas, modalidad por hoja |
| 4 | `Lista de Graduados SPB-CM-CAU_Vr.22_06_2026.xlsx` | (única) | 6 columnas (Bogotá, Medellín y VUAD) |

Las fuentes **1** y **2** comparten un esquema de 11 columnas (nombre desagregado en
apellidos y nombres). La fuente **3**, Bucaramanga, llega en **dos hojas** —una por
modalidad— con `Tipo de documento, Número de documento, Nombre, Programa, Fecha`. La
fuente **4**, SPB-CM-CAU, trae `Identificación, Nombre, Programa, Modalidad, Sede, Año`,
y su columna `sede` se mapea a etiquetas canónicas (Bogotá / Medellín / VUAD).

## Estrategia de integración

1. **Cargar** cada archivo de forma independiente.
2. **Estandarizar** cada fuente a un **esquema común** (mismas columnas y tipos), mapeando
   las columnas por posición para evitar problemas con tildes/ñ en los encabezados.
3. **Concatenar** las cuatro fuentes en un único `DataFrame` con una columna `fuente`
   que conserva la trazabilidad del origen.
4. **Normalizar y limpiar** (fecha, año, identificación, nombres de programa).
5. **Auditar la calidad** (nulos, duplicados, rangos).
6. **Analizar y visualizar** (por sede, año, programa, modalidad).
7. **Exportar** el dataset consolidado.


## 1. Configuración e importaciones

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Carpeta con los archivos fuente (relativa a la ubicación del notebook)
DATA_DIR = Path("datos")
assert DATA_DIR.exists(), f"No se encuentra la carpeta {DATA_DIR.resolve()}"

ARCHIVOS = {
    "Tunja":         DATA_DIR / "GRADUADOS UNIVERSIDAD SANTO TOMÁS SECCIONAL TUNJA.xls",
    "Villavicencio": DATA_DIR / "Base de Graduados Villavicencio para proyecto exclusivo con el programa de Estadística.xls",
    "Bucaramanga":   DATA_DIR / "GRADUADOS SANTOTO BUCARAMANGA - POSGRADO.xlsx",  # 2 hojas: PREGRADO, POSGRADO
    "SPB-CM-CAU":    DATA_DIR / "Lista de Graduados SPB-CM-CAU_Vr.22_06_2026.xlsx",
}
for k, v in ARCHIVOS.items():
    print(f"{k:14s} -> {'OK ' if v.exists() else 'FALTA'} | {v.name}")


## 2. Funciones auxiliares

Funciones de normalización reutilizables, para que la limpieza sea **explícita,
trazable y aplicada de forma idéntica** a todas las fuentes.

- `quitar_acentos` / `norm_programa`: homogeneizan los nombres de programa para poder
  agruparlos (mayúsculas, sin tildes, espacios colapsados). Se conserva además el
  texto original.
- `limpiar_identificacion`: convierte la cédula a texto y extrae los dígitos,
  preservando el valor crudo para los casos especiales (extranjería, pasaporte, etc.).
- `componer_nombre`: arma el nombre completo a partir de las cuatro columnas de nombre.


In [ ]:
def quitar_acentos(texto: str) -> str:
    '''Devuelve el texto sin tildes ni diacríticos (NFKD).'''
    if not isinstance(texto, str):
        return texto
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

def norm_programa(texto) -> str:
    '''Normaliza el nombre de un programa para agrupar: MAYÚSCULAS, sin tildes,
    espacios colapsados. Sirve como clave de agrupación, no para mostrar.'''
    if pd.isna(texto):
        return np.nan
    t = quitar_acentos(str(texto)).upper().strip()
    t = re.sub(r"\s+", " ", t)
    return t

def limpiar_identificacion(valor):
    '''Normaliza una identificación: extrae los dígitos como texto.
    Devuelve NaN si no hay dígitos. El valor original se conserva aparte.'''
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip()
    digitos = re.sub(r"\D", "", s)
    return digitos if digitos else np.nan

def componer_nombre(primer_nom, segundo_nom, primer_ape, segundo_ape) -> str:
    '''Construye 'Nombres Apellidos' a partir de las partes, ignorando vacíos.'''
    partes = [primer_nom, segundo_nom, primer_ape, segundo_ape]
    partes = [str(p).strip() for p in partes if pd.notna(p) and str(p).strip()]
    nombre = " ".join(partes)
    return re.sub(r"\s+", " ", nombre).strip() or np.nan


## 3. Esquema común

Todas las fuentes se llevan a estas columnas estándar:

| Columna | Descripción |
|---------|-------------|
| `fuente` | Origen del registro (`Tunja`, `Villavicencio`, `General`) |
| `sede` | Seccional / sede |
| `programa` | Nombre del programa (texto original) |
| `programa_norm` | Programa normalizado (clave de agrupación) |
| `modalidad` | Pregrado / Posgrado (cuando existe) |
| `tipo_identificacion` | Tipo de documento (cuando existe) |
| `identificacion` | Documento normalizado (solo dígitos) |
| `identificacion_raw` | Documento tal cual venía en la fuente |
| `nombre_completo` | Nombre y apellidos |
| `fecha_grado` | Fecha de grado (`datetime`) |
| `anio_grado` | Año de grado (entero) |


In [ ]:
COLUMNAS_ESTANDAR = [
    "fuente", "sede", "programa", "programa_norm", "modalidad",
    "tipo_identificacion", "identificacion", "identificacion_raw",
    "nombre_completo", "fecha_grado", "anio_grado",
]


## 4. Carga y estandarización por fuente

### 4.1 Fuentes con esquema de 11 columnas (Tunja y Villavicencio)

Estas dos bases comparten estructura, así que una sola función las procesa.
Se mapean las columnas **por posición** (0..10) para no depender de cómo
quedaron codificados los encabezados con tildes.


In [ ]:
def cargar_11col(ruta: Path, fuente: str, sede: str) -> pd.DataFrame:
    """Carga una base con el esquema de 11 columnas (Tunja / Villavicencio)
    y la lleva al esquema estándar. La sede se fija de forma explícita."""
    bruto = pd.read_excel(ruta)
    assert bruto.shape[1] == 11, f"Se esperaban 11 columnas en {ruta.name}, hay {bruto.shape[1]}"
    c = bruto.columns  # mapeo por posición
    p_ape, s_ape = bruto[c[5]], bruto[c[6]]
    p_nom, s_nom = bruto[c[7]], bruto[c[8]]

    out = pd.DataFrame(index=bruto.index)
    out["fuente"]              = fuente
    out["sede"]                = sede
    out["programa"]            = bruto[c[1]].astype("string").str.strip()
    out["programa_norm"]       = out["programa"].map(norm_programa)
    out["modalidad"]           = bruto[c[2]].astype("string").str.strip()
    out["tipo_identificacion"] = bruto[c[3]].astype("string").str.strip()
    out["identificacion_raw"]  = bruto[c[4]].astype("string").str.strip()
    out["identificacion"]      = out["identificacion_raw"].map(limpiar_identificacion)
    out["nombre_completo"]     = [componer_nombre(pn, sn, pa, sa)
                                  for pn, sn, pa, sa in zip(p_nom, s_nom, p_ape, s_ape)]
    out["fecha_grado"]         = pd.to_datetime(bruto[c[10]], errors="coerce")
    anio_num = pd.to_numeric(bruto[c[9]], errors="coerce")
    out["anio_grado"]          = anio_num.fillna(out["fecha_grado"].dt.year).astype("Int64")
    return out[COLUMNAS_ESTANDAR]

df_tunja = cargar_11col(ARCHIVOS["Tunja"], "Tunja", "Tunja")
df_villa = cargar_11col(ARCHIVOS["Villavicencio"], "Villavicencio", "Villavicencio")
print("Tunja        :", df_tunja.shape)
print("Villavicencio:", df_villa.shape)
df_tunja.head(3)


### 4.2 Bucaramanga (Pregrado + Posgrado, con modalidad)

Bucaramanga llega en un solo archivo con **dos hojas** —`PREGRADO` y `POSGRADO`—, cada
una con el encabezado en la segunda fila. Se cargan ambas, se etiqueta la **modalidad**
según la hoja de origen y se concatenan en una única tabla de la sede.


In [ ]:
def cargar_bucaramanga(ruta: Path) -> pd.DataFrame:
    """Carga Bucaramanga desde las hojas PREGRADO y POSGRADO (encabezado en la 2.a fila),
    con la modalidad tomada del nombre de la hoja. Esquema de columnas por posición:
    Tipo de documento, Número de documento, Nombre, Programa, Fecha."""
    partes = []
    for hoja, mod in [("PREGRADO", "Pregrado"), ("POSGRADO", "Posgrado")]:
        b = pd.read_excel(ruta, sheet_name=hoja, header=1)
        b.columns = ["tipo_id", "num_doc", "nombre", "programa", "fecha"][:len(b.columns)]
        out = pd.DataFrame(index=b.index)
        out["fuente"]              = "Bucaramanga"
        out["sede"]                = "Bucaramanga"
        out["programa"]            = b["programa"].astype("string").str.strip()
        out["programa_norm"]       = out["programa"].map(norm_programa)
        out["modalidad"]           = mod
        out["tipo_identificacion"] = b["tipo_id"].astype("string").str.strip()
        out["identificacion_raw"]  = b["num_doc"].astype("string").str.strip()
        out["identificacion"]      = out["identificacion_raw"].map(limpiar_identificacion)
        out["nombre_completo"]     = b["nombre"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
        out["fecha_grado"]         = pd.to_datetime(b["fecha"], errors="coerce")
        out["anio_grado"]          = out["fecha_grado"].dt.year.astype("Int64")
        partes.append(out[COLUMNAS_ESTANDAR])
    return pd.concat(partes, ignore_index=True)

df_buc = cargar_bucaramanga(ARCHIVOS["Bucaramanga"])
print("Bucaramanga:", df_buc.shape)
print(df_buc["modalidad"].value_counts())
df_buc.head(3)


## 5. Integración

Se concatenan las tres fuentes ya estandarizadas. La columna `fuente` mantiene la
trazabilidad del origen de cada registro.


In [ ]:
# --- Archivo SPB-CM-CAU (Sede Principal Bogotá, Campus Medellín, centros VUAD) ---
bn = pd.read_excel(ARCHIVOS["SPB-CM-CAU"])
bn.columns = ["identificacion_raw", "nombre_completo", "programa", "modalidad", "sede_det", "anio_grado"]

def sede_canonica(s):
    """Agrupa la sede a una etiqueta canónica."""
    sl = quitar_acentos(str(s)).lower().strip()
    if "bogota" in sl and "principal" in sl: return "Bogotá"
    if "medellin" in sl: return "Medellín"
    if sl.startswith("v-") or sl.startswith("v "): return "VUAD"
    return str(s).strip()

df_nuevo = pd.DataFrame(index=bn.index)
df_nuevo["fuente"]              = "SPB-CM-CAU"
df_nuevo["sede"]               = bn["sede_det"].map(sede_canonica)
df_nuevo["programa"]           = bn["programa"].astype("string").str.strip()
df_nuevo["programa_norm"]      = df_nuevo["programa"].map(norm_programa)
df_nuevo["modalidad"]          = bn["modalidad"].astype("string").str.strip()
df_nuevo["tipo_identificacion"] = pd.NA
df_nuevo["identificacion_raw"] = bn["identificacion_raw"].astype("string").str.strip()
df_nuevo["identificacion"]     = df_nuevo["identificacion_raw"].map(limpiar_identificacion)
df_nuevo["nombre_completo"]    = bn["nombre_completo"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
df_nuevo["fecha_grado"]        = pd.NaT
df_nuevo["anio_grado"]         = pd.to_numeric(bn["anio_grado"], errors="coerce").astype("Int64")
df_nuevo = df_nuevo[COLUMNAS_ESTANDAR]

# Concatenación de las cuatro fuentes (Bucaramanga + Tunja + Villavicencio + SPB-CM-CAU)
graduados = pd.concat([df_buc, df_tunja, df_villa, df_nuevo], ignore_index=True)
print("Dataset integrado:", graduados.shape)
print("\nRegistros por fuente:"); print(graduados["fuente"].value_counts())
print("\nGraduados (cédulas únicas) por sede:")
print(graduados.dropna(subset=["identificacion"]).groupby("sede")["identificacion"].nunique().sort_values(ascending=False))
print("\nCédulas únicas (población):", f"{graduados['identificacion'].nunique():,}")
graduados.sample(5, random_state=42)


## 6. Auditoría de calidad de datos

Antes de analizar, medimos completitud y consistencia: nulos por columna,
rango de fechas/años, y duplicados.


In [ ]:
print("=== Valores nulos por columna ===")
nulos = graduados.isna().sum().to_frame("n_nulos")
nulos["pct"] = (nulos["n_nulos"] / len(graduados) * 100).round(2)
print(nulos)

print("\n=== Tipos de dato ===")
print(graduados.dtypes)


In [ ]:
print("=== Rango temporal por fuente ===")
print(graduados.groupby("fuente")["anio_grado"].agg(["min", "max", "count"]))

# Años fuera de un rango plausible (control de calidad)
fuera_rango = graduados[(graduados["anio_grado"] < 1970) | (graduados["anio_grado"] > 2026)]
print(f"\nRegistros con año fuera de [1970, 2026]: {len(fuera_rango)}")
print(f"Registros sin fecha de grado válida: {graduados['fecha_grado'].isna().sum()}")
print(f"Registros sin identificación numérica: {graduados['identificacion'].isna().sum()}")


In [ ]:
# Duplicados: mismo documento + mismo programa + misma fecha => probable repetido
clave = ["identificacion", "programa_norm", "fecha_grado"]
dup_mask = graduados.dropna(subset=["identificacion"]).duplicated(subset=clave, keep=False)
n_dup = dup_mask.sum()
print(f"Registros potencialmente duplicados (doc+programa+fecha): {n_dup}")

# Personas (documento) que aparecen con más de un programa => doble titulación / posgrado
multi = (graduados.dropna(subset=["identificacion"])
         .groupby("identificacion")["programa_norm"].nunique())
print(f"Documentos con >1 programa distinto (doble titulación/posgrado): {(multi > 1).sum()}")


## 7. Análisis descriptivo

### 7.1 Graduados por fuente / sede


In [ ]:
resumen_fuente = (graduados.groupby("fuente")
                  .agg(graduados=("identificacion", "size"),
                       documentos_unicos=("identificacion", "nunique"),
                       anio_min=("anio_grado", "min"),
                       anio_max=("anio_grado", "max"))
                  )
display(resumen_fuente)

ax = graduados["fuente"].value_counts().plot(kind="bar", color=["#274690", "#1B998B", "#E07A5F", "#E8A317"])
ax.set_title("Total de registros de graduados por fuente")
ax.set_xlabel("Fuente"); ax.set_ylabel("N.º de registros")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


### 7.2 Evolución anual de graduados

In [ ]:
por_anio = (graduados.dropna(subset=["anio_grado"])
            .pivot_table(index="anio_grado", columns="fuente",
                         values="identificacion", aggfunc="size", fill_value=0)
            .sort_index())

ax = por_anio.plot(kind="line", marker="o", ms=3)
ax.set_title("Graduados por año y fuente")
ax.set_xlabel("Año de grado"); ax.set_ylabel("N.º de graduados")
ax.legend(title="Fuente")
plt.tight_layout(); plt.show()

print("Total de graduados por década:")
decada = (graduados.dropna(subset=["anio_grado"]).copy())
decada["decada"] = (decada["anio_grado"] // 10 * 10).astype("Int64").astype(str) + "s"
print(decada["decada"].value_counts().sort_index())


### 7.3 Top de programas académicos

Se usa `programa_norm` (normalizado) para agrupar correctamente pese a diferencias
de mayúsculas/tildes entre fuentes.


In [ ]:
top_prog = graduados["programa_norm"].value_counts().head(15)
ax = top_prog.sort_values().plot(kind="barh", color="#274690")
ax.set_title("Top 15 programas por número de graduados (todas las fuentes)")
ax.set_xlabel("N.º de graduados"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
top_prog.to_frame("graduados")


### 7.4 Modalidad (Pregrado vs. Posgrado)

Solo las bases de Tunja y Villavicencio traen modalidad; la base General no la
especifica (aparece como `<NA>`).


In [ ]:
mod = (graduados.assign(modalidad=graduados["modalidad"].fillna("(sin dato)"))
       .pivot_table(index="fuente", columns="modalidad",
                    values="identificacion", aggfunc="size", fill_value=0))
display(mod)

ax = mod.plot(kind="bar", stacked=True)
ax.set_title("Modalidad por fuente")
ax.set_xlabel("Fuente"); ax.set_ylabel("N.º de graduados")
plt.xticks(rotation=0); plt.legend(title="Modalidad")
plt.tight_layout(); plt.show()


## 8. Exportación del dataset consolidado

Se guarda en `salidas/`:
- **CSV** (`utf-8-sig`, compatible con Excel) para uso general.
- **Excel** `.xlsx` para revisión manual.


In [ ]:
SALIDA = Path("salidas")
SALIDA.mkdir(exist_ok=True)

ruta_csv  = SALIDA / "graduados_integrado.csv"
ruta_xlsx = SALIDA / "graduados_integrado.xlsx"

graduados.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
graduados.to_excel(ruta_xlsx, index=False)

print("Exportado:")
print(" -", ruta_csv.resolve(), f"({ruta_csv.stat().st_size/1e6:.2f} MB)")
print(" -", ruta_xlsx.resolve(), f"({ruta_xlsx.stat().st_size/1e6:.2f} MB)")
print("\nDimensiones finales:", graduados.shape)


## 9. Conclusiones y notas

- Se integraron **cuatro fuentes** en un único dataset con esquema común y trazabilidad
  de origen (`fuente`): Tunja, Villavicencio, Bucaramanga (con modalidad) y SPB-CM-CAU
  (Sede Principal de Bogotá, Campus Medellín y centros VUAD).
- Con la incorporación de SPB-CM-CAU el universo cubre las **seis sedes** de la
  Universidad; la base resultante son **197.273 registros** de grado de **174.685
  personas** (cédulas únicas), con cobertura entre **1970 y 2026**.
- La **identificación** se normalizó a solo dígitos conservando el valor original
  (`identificacion_raw`) para no perder los casos de extranjería/pasaporte.
- Los nombres de programa se agrupan vía `programa_norm` para evitar la fragmentación
  por tildes o mayúsculas.

### Limitaciones / siguientes pasos
- **Solapamiento entre sedes**: una misma persona puede tener títulos en más de una sede
  (3.514 casos), de modo que la suma de graduados por sede supera el total de personas
  únicas; la unidad poblacional es la cédula.
- Los **duplicados** detectados en la sección 6 deben revisarse según la regla de negocio
  (¿una persona con dos programas equivale a dos graduaciones válidas?).
- Enriquecer con catálogos oficiales de programas/facultades si se requiere un reporte
  institucional.
